In [2]:
# SoRL in modded-gpt compatible fashion (for ultra fast pre-training)
# 1. pre-training demands simple model architecture, even .generate function can be wrapped around the trained model afterwards
# 2. no need to include 'kv-cache' for the pre-training experiment here

In [ ]:
from sorl.model import CausalSelfAttention, Block, GPTConfig
import torch 

# mock input 
x = torch.randn(2, 1024, 768)

config = GPTConfig()
attn = CausalSelfAttention(dim=768, n_head=6)
block = Block(config=config)

y, v1 = attn(x)
x, v1 = block(x, v1, x, None)

In [3]:
import torch 
from sorl.gat import GATConfig, GAT

gat_config = GATConfig(vocab_sizes=[128,8],
          n_layer=12,
          n_head=6,
          n_embd=768,
          flex_kernel_options=None)

model = GAT(gat_config)


token_ids = torch.randint(0, 128 + 8, (2, 4))
idx = token_ids[:, :-1].contiguous()
target = token_ids[:, 1:].contiguous()


# forward pass ()
ppt = model(idx, target, 1024)

In [4]:
from sorl.gat import parallel_denoise
from sorl.gat import generate 


num_iterations = 5 
memory_span = 1024 
temperature = 0.0

parallel_denoise(model, idx, num_iterations=5, memory_span=1024, temperature=0.0)

generate(model, idx, max_new_tokens=5, abstraction_interval=3)


tensor([[ 42,  86, 110, 129,   0,   0, 129,   0],
        [ 46,  21,   4, 129,   0,   0, 129,   0]])

In [24]:
idx = torch.tensor([
    [4, 129, 129, 129],
    [3, 10, 129, 129]
])

In [1]:
import torch 
from sorl.gat_act import GATConfig, GAT

gat_config = GATConfig(vocab_sizes=[128,8],
          n_layer=12,
          n_head=6,
          n_embd=768,
          flex_kernel_options=None)

model = GAT(gat_config)

from sorl.gat_act import infer_level

# idx = torch.randint(0, model.vocab_sizes.sum(), (2, 4)).contiguous()
idx = torch.tensor([
    [4, 129, 130, 129],
    [3, 10, 130, 129]
]).contiguous()
levels = infer_level(idx, model.vocab_sizes)
abstract_mask = (levels > 0)

In [4]:
from sorl.gat_act import search, generate

# search(model, idx, max_iterations=1, n_continuous=0, memory_span=1024, temperature=0.0, K=3)

# Issue 1. generate method keeps producing abstract tokens
generate(model, idx, max_new_tokens=10, K=3)

tensor([[  4, 129, 129, 129,   0,   0, 129,   0,   0, 129,   0,   0, 129,   0],
        [  3,  10, 129, 129,   0,   0, 129,   0,   0, 129,   0,   0, 129,   0]])

In [50]:
# Continuous recursion & Discrete recursion
# -------------------------------------------
# Remark 1. 'abstract_mask' is more of a 'recursion_mask' -- of course we only recurse on abstract tokens
# Reflection 1. We use 'recursion_mask' only for denoising process & forward pass
# 'Denoise' method basically does discrete recursion, we should be more upfront about this in terminology
# Idea 1. perhaps the explicit separation between continuous recursion & discrete recursion is okay? we could 
#         experiment both individually, and perhaps asynchronous recuring on them makes sense ...
# Reflection 2. When generate new tokens, we'd also recurse things simultaneously, we almost need a 'forward_with_recursion' method
#               that does discrete & continuous recursion simultaneously, at least we should not waste the next token prediction logits
#               (for instance, every recursion leads to a different next-token prediction)
# Reflection 3. From reflection-2, it seems that there is no real difference between 'generate' method and 'recursion' method
#               since generate is basically just 'recursion' + decode on last token representation


# it does feels like we can simplify it. 
# as what I need is a 'recursion' method, that perform some pre-determined # of continuous recursion (via passing and using abstract_repr), as well as some pre-determined # of discrete recursion (via passing and re-using the deocded abstract tokens)

# here a reflection 4. is that there is no reason why ACT is only done on the continuous recursion, we ought to try ACT on both continuous recursion and discrete recursion --- according to the lesson from TRM/HRM work, ACT on discrete recursion is probably more meaningful



# Reflection 1. 
# - continuous recursion assumes invariant recursion tokens, rep + wte = new_wte
# - discrete recursion changes recursion tokens, this makes inner-outer loop more sensible than joint loop

# Reflection 2. 
# - to mimic TRM/HRM settings, we could separate inner/outer loop
# - inner loop contains fixed # of continuous recursion
# - outer loop does ACT & discrete recursion



In [62]:
import torch.nn.functional as F
from sorl.gat_act import get_logits_mask, _continuous_recursion_step, _discrete_recursion_step, compute_act

def get_next_token_level(seq_length, abstraction_interval):
    # assumes L=2 (to be extended)
    return 1 if (seq_length % abstraction_interval == 0) else 0

def rythmic_decode(logits, model, K, temperature=0.0): 

    nt_logits = logits[:, -1]
    level = get_next_token_level(logits.shape[-1], K)
    logit_mask = get_logits_mask(level, model.vocab_sizes)

    nt_logits = torch.where(
        logit_mask.expand_as(nt_logits), 
        nt_logits, 
        torch.tensor(float('-inf'), device=nt_logits.device)
    )

    if temperature == 0.0: 
        next_token_id = nt_logits.argmax(dim=-1, keepdim=True)
    else: 
        probs = F.softmax(nt_logits / temperature, dim=-1)
        next_token_id = torch.multinomial(probs, num_samples=1)

    return next_token_id

# generate function

max_iterations = 5 
memory_span = 1024 
temperature = 0.0
n_continuous = 0
K = 4 

def search(model, idx, max_iterations, n_continuous, memory_span, temperature, K):
    """Recursion without loss computation"""
    idx = idx.clone() 
        
    levels = infer_level(idx, model.vocab_sizes)
    abs_mask = levels > 0 
    abs_mask[:, 0] = False
    recursion_mask = abs_mask.clone()

    abstract_repr = torch.zeros(abs_mask.sum(), model.n_embd, device=idx.device)
    for iteration in range(max_iterations): 
        for _ in range(n_continuous): 
            abstract_repr, x = _continuous_recursion_step(
                model, idx, abstract_repr, abs_mask, recursion_mask, memory_span
            )

        x = model._forward_pass(idx, abstract_repr, abs_mask, memory_span)
        logits = model._compute_logits(x)   
        idx = _discrete_recursion_step(model, idx, logits, recursion_mask, temperature)
        
        act = compute_act(logits, idx, abs_mask)

        recursion_mask = recursion_mask & ~act.unsqueeze(1)
        if not recursion_mask.any():
            break
    
    return idx, logits

def generate(model, idx, max_new_tokens=50, abstraction_interval=4, 
             max_iterations=5, n_continuous=0, memory_span=1024, temperature=0.0, K=4):
  
    idx = idx.clone()

    with torch.no_grad(): 
        for _ in range(max_new_tokens): 
            # --- search --- 
            idx, logits = search(model, idx, max_iterations, n_continuous, memory_span, temperature, K)

            # --- decode --- 
            next_token_id = rythmic_decode(logits, model, K, temperature)
            idx = torch.cat((idx, next_token_id), dim=1)

    return idx

